In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE_DIR = Path("/Data/in Use")
ROUNDS_PATH = BASE_DIR / "combined_rounds_all_2017_2026.csv"
EVENT_SKILL_PATH = BASE_DIR / "event_skill.xlsx"

combined = pd.read_csv(
    ROUNDS_PATH,
    dtype={"round_date": "string"},
    low_memory=False,
)

event_skill = pd.read_excel(EVENT_SKILL_PATH)

if "event_date" in event_skill.columns:
    event_skill["event_date"] = pd.to_datetime(event_skill["event_date"], errors="coerce")

event_skill_small = event_skill[["year", "event_id", "avg_skill"]].drop_duplicates()

combined = combined.merge(
    event_skill_small,
    on=["year", "event_id"],
    how="left",
)

combined["avg_skill"] = pd.to_numeric(combined["avg_skill"], errors="coerce")
combined["year"] = pd.to_numeric(combined["year"], errors="coerce")
combined["sg_total"] = pd.to_numeric(combined["sg_total"], errors="coerce")
combined["round_score"] = pd.to_numeric(combined["round_score"], errors="coerce")

combined["avg_skill"] = combined["avg_skill"].fillna(1.0)

In [3]:
def _weighted_mean(values, weights):
    mask = (~pd.isna(values)) & (~pd.isna(weights))
    if not mask.any():
        return float("nan")
    v = values[mask]
    w = weights[mask]
    return (v * w).sum() / w.sum()

In [4]:
MIN_ROUNDS_HIST = 50
LAMBDA_YEARS = 0.35
K_FIELD = 0.25

def save_preseason(df: pd.DataFrame, target_season: int, base_path: str):
    path = Path(base_path) / f"preseason_{target_season}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")

def build_preseason_history_star_power_style(
    combined: pd.DataFrame,
    target_season: int,
    min_rounds: int = MIN_ROUNDS_HIST,
    lambda_years: float = LAMBDA_YEARS,
    k_field: float = K_FIELD,
) -> pd.DataFrame:

    prev1 = target_season - 1
    prev2 = target_season - 2

    df = combined.copy()
    df["tour"] = df["tour"].astype(str).str.upper()
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["sg_total"] = pd.to_numeric(df["sg_total"], errors="coerce")
    df["round_score"] = pd.to_numeric(df["round_score"], errors="coerce")
    df["avg_skill"] = pd.to_numeric(df["avg_skill"], errors="coerce")
    df["event_completed"] = pd.to_datetime(df["event_completed"], errors="coerce")

    pga_players_prev1 = (
        df[(df["year"] == prev1) & (df["tour"] == "PGA")]["dg_id"]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    hist = df[df["year"].isin([prev2, prev1])].copy()
    hist = hist[hist["dg_id"].isin(pga_players_prev1)]

    if hist.empty:
        return pd.DataFrame(
            columns=[
                "dg_id",
                "target_season",
                "total_rounds_hist",
                "raw_sg",
                "wgh_sg",
                "dec_raw_sg",
                "dec_wgh_sg",
                "avg_round_score",
            ]
        )

    hist["W_raw"] = 1 + k_field * hist["avg_skill"]

    pga_mask = (hist["tour"] == "PGA") & hist["W_raw"].notna()
    pga_mean = hist.loc[pga_mask, "W_raw"].mean()
    if pd.isna(pga_mean) or pga_mean == 0:
        pga_mean = 1.0

    hist["W_field"] = (hist["W_raw"] / pga_mean).fillna(1.0)

    hist["sg_total_weighted"] = hist["sg_total"] * hist["W_field"]

    max_date = hist["event_completed"].max()
    age_days = (max_date - hist["event_completed"]).dt.days
    age_years = age_days / 365.25
    hist["recency_weight"] = np.exp(-lambda_years * age_years)

    hist["sg_total_rec_num"] = hist["sg_total"] * hist["recency_weight"]
    hist["sg_total_wgh_rec_num"] = hist["sg_total_weighted"] * hist["recency_weight"]

    group_cols = ["dg_id"]

    grp = (
        hist.groupby(group_cols)
        .agg(
            total_rounds_hist=("sg_total", "count"),

            raw_sg=("sg_total", "mean"),
            wgh_sg=("sg_total_weighted", "mean"),
            avg_round_score=("round_score", "mean"),

            dec_raw_num=("sg_total_rec_num", "sum"),
            dec_wgh_num=("sg_total_wgh_rec_num", "sum"),
            recency_w_sum=("recency_weight", "sum"),

            player_name_latest=("player_name", "last"),
        )
        .reset_index()
    )

    grp["dec_raw_sg"] = grp["dec_raw_num"] / grp["recency_w_sum"].replace(0, np.nan)
    grp["dec_wgh_sg"] = grp["dec_wgh_num"] / grp["recency_w_sum"].replace(0, np.nan)

    grp = grp[grp["total_rounds_hist"] >= min_rounds].copy()

    grp["target_season"] = target_season

    grp = grp[
        [
            "dg_id",
            "target_season",
            "player_name_latest",
            "total_rounds_hist",
            "raw_sg",
            "wgh_sg",
            "dec_raw_sg",
            "dec_wgh_sg",
            "avg_round_score",
        ]
    ].sort_values("dec_wgh_sg", ascending=False)

    return grp

In [5]:
# pre_2024 = build_preseason_history_star_power_style(combined, 2024)
# save_preseason(pre_2024, 2024, "/Users/joshmacbook/python_projects/OAD/Data/in Use/")
#
# pre_2025 = build_preseason_history_star_power_style(combined, 2025)
# save_preseason(pre_2025, 2025, "/Users/joshmacbook/python_projects/OAD/Data/in Use/")
#
# pre_2024 = build_preseason_history_star_power_style(combined, 2024)
# pre_2025 = build_preseason_history_star_power_style(combined, 2025)
#
# pre_2024.head(500)

pre_2026 = build_preseason_history_star_power_style(combined, 2026)
save_preseason(pre_2026, 2026, "/Data/in Use/")

pre_2026.head(500)

Saved: /Users/joshmacbook/python_projects/OAD/Data/in Use/preseason_2026.csv


,dg_id,target_season,player_name_latest,total_rounds_hist,raw_sg,wgh_sg,dec_raw_sg,dec_wgh_sg,avg_round_score
376,18417,2026,"Scheffler, Scottie",167,2.731994,2.743233,2.754110,2.747771,67.880240
409,19195,2026,"Rahm, Jon",129,2.248171,2.211731,2.260259,2.177122,68.527132
660,25680,2026,"Yu-cheng, Hsu",51,1.900314,2.111660,1.903153,2.114815,74.235294
441,19841,2026,"DeChambeau, Bryson",108,1.904444,1.872905,1.926673,1.861792,68.898148
102,10091,2026,"McIlroy, Rory",187,1.870845,1.867322,1.881844,1.846015,68.967213
...,...,...,...,...,...,...,...,...,...
64,7881,2026,"Kang, Sunghoon",68,-1.547618,-1.518510,-1.657683,-1.581950,72.485294
48,7305,2026,"Gutschewski, Scott",53,-1.714377,-1.864547,-1.706513,-1.844071,71.849057
428,19620,2026,"Pereda, Raul",58,-1.909948,-2.075295,-1.936239,-2.077383,72.137931
276,15634,2026,"Trainer, Martin",88,-2.658386,-2.212531,-2.878646,-2.306759,73.193182


**Function: SG-based course fit profiles (DG-inspired)**

In [6]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression


def build_course_fit_5attr(
    combined: pd.DataFrame,
    target_season: int,
    max_rounds: int = 150,
    min_rounds: int = 20,
    min_obs_per_course: int = 80,
    shrink_k: float = 400.0,
) -> pd.DataFrame:
    """
    DG-style 5-attribute course fit for all PGA courses played in `target_season`,
    using ONLY data from seasons < target_season.

    Attributes used (skills):
      - Driving Distance  -> driving_dist
      - Driving Accuracy  -> driving_acc
      - SG Approach       -> sg_app
      - SG Around Green   -> sg_arg
      - SG Putting        -> sg_putt

    Steps:
      1) Build per-player baseline skills for the 5 attributes from all PGA rounds
         with season < target_season (exp-decayed over last `max_rounds`).
      2) Build event-level performance (sum sg_total) for those historical events.
      3) Center event performance within each event-course.
      4) Globally standardize the 5 skills (z-scores).
      5) For each PGA course used in target_season:
           - subset historical event-player rows for that course
           - require >= min_obs_per_course rows
           - regress event_sg_total_centered on the 5 z-skill attributes
           - convert abs(coefs) -> normalized importances
           - apply shrinkage n_obs / (n_obs + shrink_k)
      6) Return a tidy DataFrame with course-level importances.

    Returns:
      DataFrame columns:
        course_num, course_name,
        imp_dist, imp_acc, imp_app, imp_arg, imp_putt,
        n_obs, n_events, n_players, target_season
    """

    # -----------------------------
    # 0. Basic cleaning / typing
    # -----------------------------
    df = combined.copy()

    df["tour"] = df["tour"].astype(str).str.upper()
    df["season"] = pd.to_numeric(df["season"], errors="coerce")
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["dg_id"] = pd.to_numeric(df["dg_id"], errors="coerce").astype("Int64")
    df["event_completed"] = pd.to_datetime(df["event_completed"], errors="coerce")

    # SG + driving columns numeric
    for col in ["sg_app", "sg_arg", "sg_putt", "sg_total", "driving_dist", "driving_acc"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        else:
            raise ValueError(f"Required column '{col}' not found in combined DataFrame.")

    # -----------------------------
    # 1. Historical PGA rounds
    # -----------------------------
    hist = df[(df["tour"] == "PGA") & (df["season"] < target_season)].copy()

    if hist.empty:
        raise ValueError(f"No historical PGA rounds with season < {target_season}.")

    # Use only rows where we have all 5 attributes and sg_total
    attr_cols = ["driving_dist", "driving_acc", "sg_app", "sg_arg", "sg_putt", "sg_total"]
    hist = hist.dropna(subset=attr_cols + ["dg_id", "event_completed"])

    # Order by player and time for baseline computation
    hist = (
        hist.sort_values(
            ["dg_id", "event_completed", "event_id", "round_num"],
            ascending=[True, True, True, True],
        )
        .reset_index(drop=True)
    )

    # -----------------------------
    # 2. Build per-player exp-decay baselines for 5 attributes
    # -----------------------------
    # Half-lives in *rounds* (you can tweak these later if you want)
    half_lives = {
        "driving_dist": 25,   # distance stabilizes quickly
        "driving_acc": 25,    # accuracy similar
        "sg_app": 60,         # approach / arg moderate horizon
        "sg_arg": 60,
        "sg_putt": 120,       # putting very noisy -> long horizon
    }

    def _agg_player(g: pd.DataFrame) -> pd.Series:
        """
        Compute exp-decay skill for one player over all historical rounds.
        Works whether 'dg_id' is still a column in g or only available as g.name.
        """
        # get dg_id from column if present, otherwise from group key
        if "dg_id" in g.columns:
            dg_id_val = g["dg_id"].iloc[0]
        else:
            dg_id_val = g.name

        g = g.copy().sort_values("event_completed", ascending=False)
        g = g.head(max_rounds)

        n = len(g)
        out = {
            "dg_id": dg_id_val,
            "player_name": g["player_name"].iloc[0],
            "n_rounds": n,
        }

        if n < min_rounds:
            # Not enough data -> mark skills as NaN; they'll drop out later
            for col in ["skill_dist", "skill_acc", "skill_app", "skill_arg", "skill_putt"]:
                out[col] = np.nan
            return pd.Series(out)

        idx = np.arange(n)

        # For each stat, compute exp-decayed mean
        for stat, hl in half_lives.items():
            lam = np.log(2) / hl  # half-life in rounds
            w = np.exp(-lam * idx)
            w = w / w.sum()

            vals = g[stat].to_numpy()
            skill = float(np.sum(w * vals))
            key = {
                "driving_dist": "skill_dist",
                "driving_acc": "skill_acc",
                "sg_app": "skill_app",
                "sg_arg": "skill_arg",
                "sg_putt": "skill_putt",
            }[stat]
            out[key] = skill

        return pd.Series(out)

    skills = (
        hist.groupby("dg_id", group_keys=False)
            .apply(_agg_player, include_groups=False)
            .reset_index(drop=True)
    )

    # Keep only players with valid skills (all 5 non-null)
    skill_cols = ["skill_dist", "skill_acc", "skill_app", "skill_arg", "skill_putt"]
    skills = skills.dropna(subset=skill_cols)
    if skills.empty:
        raise ValueError("No players with valid 5-attribute skills for course fit.")

    # -----------------------------
    # 3. Build historical event-level performance and merge skills
    # -----------------------------
    event_perf = (
        hist.groupby(
            ["event_id", "event_name", "course_num", "course_name", "dg_id"],
            as_index=False,
        )
        .agg(event_sg_total=("sg_total", "sum"))
    )

    # Merge player skills
    event_perf = event_perf.merge(
        skills[["dg_id"] + skill_cols],
        on="dg_id",
        how="inner",
    )

    if event_perf.empty:
        raise ValueError("No event-player rows after merging skills; check data coverage.")

    # Center event_sg_total by event+course (performance vs field)
    event_perf["event_sg_total_centered"] = (
        event_perf["event_sg_total"]
        - event_perf.groupby(["event_id", "course_num"])["event_sg_total"].transform("mean")
    )

    # -----------------------------
    # 4. Standardize skills globally (z-scores)
    # -----------------------------
    z_cols = {}
    for col, zname in zip(
        skill_cols,
        ["z_dist", "z_acc", "z_app", "z_arg", "z_putt"],
    ):
        mu = event_perf[col].mean()
        sd = event_perf[col].std(ddof=0)
        if sd == 0 or np.isnan(sd):
            event_perf[zname] = 0.0
        else:
            event_perf[zname] = (event_perf[col] - mu) / sd
        z_cols[col] = zname

    # -----------------------------
    # 5. Identify PGA courses in target_season (course_num-based)
    # -----------------------------
    # courses_target = (
    #     df[(df["tour"] == "PGA") & (df["season"] == target_season)]
    #     .loc[:, ["course_num", "course_name"]]
    #     .dropna(subset=["course_num"])
    #     .drop_duplicates()
    #     .sort_values("course_num")
    # )
    #
    # if courses_target.empty:
    #     raise ValueError(f"No PGA courses found for season {target_season}.")

    # -----------------------------
    # 6. Per-course regression to get importances
    # -----------------------------
    profiles = []

    for _, row in courses_target.iterrows():
        cnum = row["course_num"]
        cname = row["course_name"]

        df_c = event_perf[event_perf["course_num"] == cnum].copy()
        if len(df_c) < min_obs_per_course:
            # not enough data to say anything meaningful
            continue

        df_c = df_c.dropna(
            subset=[
                "event_sg_total_centered",
                "z_dist", "z_acc", "z_app", "z_arg", "z_putt",
            ]
        )
        if len(df_c) < min_obs_per_course:
            continue

        X = df_c[["z_dist", "z_acc", "z_app", "z_arg", "z_putt"]].values
        y = df_c["event_sg_total_centered"].values

        model = LinearRegression()
        model.fit(X, y)
        coefs = model.coef_  # [dist, acc, app, arg, putt]

        abs_coefs = np.abs(coefs)
        s = abs_coefs.sum()
        if s == 0:
            imp_dist = imp_acc = imp_app = imp_arg = imp_putt = np.nan
        else:
            imp_vals = abs_coefs / s
            imp_dist, imp_acc, imp_app, imp_arg, imp_putt = imp_vals.tolist()

        n_obs = len(df_c)
        # shrinkage toward neutrality (helps small-sample courses)
        shrink = n_obs / (n_obs + shrink_k)
        imp_dist *= shrink
        imp_acc *= shrink
        imp_app *= shrink
        imp_arg *= shrink
        imp_putt *= shrink

        profiles.append(
            {
                "course_num": cnum,
                "course_name": cname,
                "imp_dist": imp_dist,
                "imp_acc": imp_acc,
                "imp_app": imp_app,
                "imp_arg": imp_arg,
                "imp_putt": imp_putt,
                "n_obs": n_obs,
                "n_events": df_c["event_id"].nunique(),
                "n_players": df_c["dg_id"].nunique(),
                "target_season": target_season,
            }
        )

    profiles_df = pd.DataFrame(profiles)
    if not profiles_df.empty:
        profiles_df = profiles_df.sort_values("course_num").reset_index(drop=True)

    return profiles_df

In [7]:
ROUNDS_PATH = "/Data/in Use/combined_rounds_all_2017_2026.csv"
combined = pd.read_csv(ROUNDS_PATH, low_memory=False)

# course_fit_2024 = build_course_fit_5attr(combined, target_season=2024)
# course_fit_2025 = build_course_fit_5attr(combined, target_season=2025)
course_fit_2025 = build_course_fit_5attr(combined, target_season=2026)

base_dir = "/Data/in Use"
# course_fit_2024.to_csv(f"{base_dir}/course_fit_2024_dg_style_5attr.csv", index=False)
# course_fit_2025.to_csv(f"{base_dir}/course_fit_2025_dg_style_5attr.csv", index=False)
course_fit_2025.to_csv(f"{base_dir}/course_fit_2026_dg_style_5attr.csv", index=False)


ValueError: No PGA courses found for season 2026.

In [66]:
import pandas as pd

path = "/Data/in Use/combined_rounds_all_2017_2026.csv"
df2 = pd.read_csv(path, low_memory=False)
print(df2.columns.tolist())


['tour', 'year', 'season', 'event_completed', 'event_name', 'event_id', 'player_name', 'dg_id', 'fin_text', 'round_num', 'course_name', 'course_num', 'course_par', 'start_hole', 'teetime', 'round_score', 'sg_putt', 'sg_arg', 'sg_app', 'sg_ott', 'sg_t2g', 'sg_total', 'driving_dist', 'driving_acc', 'gir', 'scrambling', 'prox_rgh', 'prox_fw', 'great_shots', 'poor_shots', 'eagles_or_better', 'birdies', 'pars', 'bogies', 'doubles_or_worse', 'finish_num', 'round_date']


In [5]:
import pandas as pd

from Scripts.Archive.odds_results import load_odds_and_results
from Scripts.Archive.oad_schedule import load_oad_for_season

season = 2024  # or 2025

# ----- Odds side -----
odds = load_odds_and_results(copy=True)
odds["year"] = pd.to_numeric(odds["year"], errors="coerce").astype("Int64")
odds = odds[odds["year"] == season].copy()

if "event_id" not in odds.columns:
    print("Odds file is missing 'event_id' column; columns are:", odds.columns.tolist())
else:
    odds["event_id"] = pd.to_numeric(odds["event_id"], errors="coerce").astype("Int64")

print("Unique (year, event_id) from Odds for", season)
print(
    odds[["year", "event_id", "event_name"]]
        .drop_duplicates()
        .sort_values(["year", "event_id"])
        .head(50)
)

# ----- OAD side -----
oad = load_oad_for_season(season)

print("\nUnique (year, event_id) from OAD for", season)
print(
    oad[["year", "event_id", "event_name", "purse"]]
        .drop_duplicates()
        .sort_values(["year", "event_id"])
        .head(50)
)

# ----- Join check -----
join_check = (
    odds[["year", "event_id", "event_name"]]
    .drop_duplicates()
    .merge(
        oad[["year", "event_id", "event_name", "purse"]],
        on=["year", "event_id"],
        how="left",
        suffixes=("", "_oad"),
    )
)

print("\nRows where purse is missing after simple join:")
print(
    join_check[join_check["purse"].isna()]
        .sort_values(["year", "event_id"])
        .head(50)
)


Unique (year, event_id) from Odds for 2024
      year  event_id                                         event_name
4382  2024         2                               The American Express
4773  2024         3                                    WM Phoenix Open
4537  2024         4                             Farmers Insurance Open
4693  2024         5                           AT&T Pebble Beach Pro-Am
4238  2024         6                                Sony Open in Hawaii
4904  2024         7                           The Genesis Invitational
5247  2024         9  Arnold Palmer Invitational presented by Master...
5104  2024        10              Cognizant Classic in The Palm Beaches
5316  2024        11                           THE PLAYERS Championship
5997  2024        12                                       RBC Heritage
7803  2024        13                               Wyndham Championship
5908  2024        14                                 Masters Tournament
4179  2024        16 

In [6]:
from Scripts.Archive.ev_utils import add_ev_for_events

ev_2024 = add_ev_for_events(2024)

cols = ["year", "event_id", "event_name", "player_name", "decimal_odds_std", "implied_prob", "purse", "ev_raw"]

ev_2024[cols].query("event_id == 11").head(10)  # THE PLAYERS


EV WARNING: These events still have no purse after fallback merge:
 year  event_id event_name
 2024        16 The Sentry


,year,event_id,event_name,player_name,decimal_odds_std,implied_prob,purse,ev_raw
1137,2024,11,THE PLAYERS Championship,"Scheffler, Scottie",6.5,0.153846,25000000.0,3.846154e+06
1138,2024,11,THE PLAYERS Championship,"Fitzpatrick, Matt",76.0,0.013158,25000000.0,3.289474e+05
1139,2024,11,THE PLAYERS Championship,"Putnam, Andrew",121.0,0.008264,25000000.0,2.066116e+05
1140,2024,11,THE PLAYERS Championship,"Woodland, Gary",401.0,0.002494,25000000.0,6.234414e+04
1141,2024,11,THE PLAYERS Championship,"Mitchell, Keith",101.0,0.009901,25000000.0,2.475248e+05
1142,2024,11,THE PLAYERS Championship,"Aberg, Ludvig",36.0,0.027778,25000000.0,6.944444e+05
1143,2024,11,THE PLAYERS Championship,"Montgomery, Taylor",351.0,0.002849,25000000.0,7.122507e+04
1144,2024,11,THE PLAYERS Championship,"Dahmen, Joel",401.0,0.002494,25000000.0,6.234414e+04
1145,2024,11,THE PLAYERS Championship,"Conners, Corey",61.0,0.016393,25000000.0,4.098361e+05
1146,2024,11,THE PLAYERS Championship,"Bezuidenhout, Christiaan",151.0,0.006623,25000000.0,1.655629e+05


In [7]:
from Scripts.Archive.ev_utils import add_ev_for_events

ev_2024 = add_ev_for_events(2024)
ev_2024[["year", "event_id", "event_name", "purse", "ev_raw"]] \
    .drop_duplicates(subset=["year", "event_id"]) \
    .sort_values(["year", "event_id"])


EV WARNING: These events still have no purse after fallback merge:
 year  event_id event_name
 2024        16 The Sentry


,year,event_id,event_name,purse,ev_raw
203,2024,2,The American Express,8400000.0,2.393162e+04
594,2024,3,WM Phoenix Open,8800000.0,5.827815e+04
358,2024,4,Farmers Insurance Open,9000000.0,6.870229e+04
514,2024,5,AT&T Pebble Beach Pro-Am,20000000.0,3.278689e+05
59,2024,6,Sony Open in Hawaii,8300000.0,2.364672e+04
725,2024,7,The Genesis Invitational,20000000.0,2.816901e+05
1068,2024,9,Arnold Palmer Invitational presented by Master...,20000000.0,2.666667e+06
925,2024,10,Cognizant Classic in The Palm Beaches,9000000.0,6.870229e+04
1137,2024,11,THE PLAYERS Championship,25000000.0,3.846154e+06
1818,2024,12,RBC Heritage,20000000.0,3.636364e+06


In [8]:
ev_2024.query("event_id == 16")[["event_name", "purse", "ev_raw"]].head()
ev_2024.query("event_id == 525")[["event_name", "purse", "ev_raw"]].head()


,event_name,purse,ev_raw
3469,3M Open,8100000.0,114084.507042
3470,3M Open,8100000.0,122727.272727
3471,3M Open,8100000.0,197560.975610
3472,3M Open,8100000.0,80198.019802
3473,3M Open,8100000.0,80198.019802


In [16]:
import pandas as pd

path = "/Data/in Use/preseason_2024.csv"
test = pd.read_csv(path)
print(test.columns.tolist())

['dg_id', 'hist_rounds', 'mean_sg_total', 'mean_sg_total_field', 'dec_sg_total', 'dec_sg_total_field', 'mean_round_score', 'player_name_latest', 'target_season']


In [1]:
import pandas as pd
from pathlib import Path

# ----------------------------
# Paths
# ----------------------------
DATA_PATH = Path(
    "/Data/in Use/combined_rounds_all_2017_2026.csv"
)

# ----------------------------
# Load data
# ----------------------------
df = pd.read_csv(DATA_PATH)

# ----------------------------
# Identify season/year column
# ----------------------------
if "season" in df.columns:
    year_col = "season"
elif "year" in df.columns:
    year_col = "year"
else:
    raise ValueError("No 'season' or 'year' column found")

# ----------------------------
# Filter to 2025
# ----------------------------
df_2025 = df[df[year_col] == 2025].copy()

# ----------------------------
# Validate required columns
# ----------------------------
required_cols = {"event_id", "event_name"}
missing = required_cols - set(df_2025.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# ----------------------------
# Get unique events
# ----------------------------
events_2025 = (
    df_2025[["event_id", "event_name"]]
    .dropna(subset=["event_id"])
    .drop_duplicates()
    .sort_values("event_id")
    .reset_index(drop=True)
)

# ----------------------------
# Output
# ----------------------------
print(events_2025)

# Optional: save to CSV
out_path = DATA_PATH.parent / "events_2025_unique.csv"
events_2025.to_csv(out_path, index=False)

print(f"\nSaved to: {out_path}")


     event_id                                        event_name
0           2                              The American Express
1           3                                   WM Phoenix Open
2           4                            Farmers Insurance Open
3           5                          AT&T Pebble Beach Pro-Am
4           6                               Sony Open in Hawaii
..        ...                                               ...
107   2025140                             Turkish Airlines Open
108   2025141                       DP World India Championship
109   2025142                     Commercial Bank Qatar Masters
110   2025143  Austrian Alpine Open presented by SalzburgerLand
111   2025715            DP World Tour Qualifying (Final Stage)

[112 rows x 2 columns]

Saved to: /Users/joshmacbook/python_projects/OAD/Data/in Use/events_2025_unique.csv


/var/folders/85/gv9dnstn1tn96gql2f7mj15h0000gn/T/ipykernel_6160/4231154544.py:14: DtypeWarning: Columns (36) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)
